# StandUp4AI: 1000-Video Evaluation Pipeline

NO rclone — Drive is mounted locally, so we use pathlib directly.
Run all cells top to bottom. GPU runtime not required (feature extraction is CPU).

In [ ]:
# Cell 1: Mount Drive + setup (NO rclone — plain filesystem access)
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
from pathlib import Path

BASE = "/content/drive/MyDrive/standup4ai"
AUDIO_DIRS = [f"{BASE}/audio", f"{BASE}/audio_1000"]   # both folders
LABELS_DIR = f"{BASE}/labels"                          # searched recursively
OUT_DIR = f"{BASE}/eval_1000"
PARTITION = f"{BASE}/standup4ai_partition.csv"
os.makedirs(OUT_DIR, exist_ok=True)

print("Audio:", AUDIO_DIRS)
print("Labels:", LABELS_DIR)
print("Partition:", PARTITION)
print("Output:", OUT_DIR)

In [ ]:
# Cell 2: Scan Drive with pathlib (replaces ALL rclone calls)
# Audio files from BOTH folders
audio_files = {}  # vid -> full path
for d in AUDIO_DIRS:
    p = Path(d)
    if not p.exists():
        print(f"[warn] missing dir: {d}")
        continue
    for f in p.glob('*.m4a'):
        audio_files.setdefault(f.stem, str(f))
    for f in p.glob('*.mp3'):
        audio_files.setdefault(f.stem, str(f))
    for f in p.glob('*.wav'):
        audio_files.setdefault(f.stem, str(f))
print(f"Audio files: {len(audio_files)}")

# Label CSVs — recursive (handles train/val/test subfolders OR flat)
label_files = {}  # vid -> full path
lp = Path(LABELS_DIR)
if lp.exists():
    for f in lp.rglob('*.csv'):
        label_files.setdefault(f.stem, str(f))
else:
    print(f"[warn] missing labels dir: {LABELS_DIR}")
print(f"Label files: {len(label_files)}")

# Partition
df = pd.read_csv(PARTITION)
col_part = df.columns[2] if len(df.columns) >= 3 else None
val_ids = set(df[df['part'] == 'val']['fn'])
train_ids = set(df[df['part'] == 'train']['fn'])
print(f"Val: {len(val_ids)}, Train: {len(train_ids)}")

# Eval-ready = partition ID has BOTH audio AND label file
val_eval = val_ids & audio_files.keys() & label_files.keys()
train_eval = train_ids & audio_files.keys() & label_files.keys()
print(f"Val eval-ready: {len(val_eval)}")
print(f"Train eval-ready: {len(train_eval)}")
print(f"Total eval-ready: {len(val_eval) + len(train_eval)}")

assert len(val_eval) + len(train_eval) > 0, "No eval-ready videos — check folders above!"

In [ ]:
# Cell 3: Feature extraction (15-dim prosody — same as training)
import librosa
import warnings
warnings.filterwarnings('ignore')

def extract_features(audio_path, sr=22050):
    y, sr = librosa.load(audio_path, sr=sr, mono=True)
    if len(y) < 0.5 * sr:
        return None
    spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    spec_bw = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    spec_roll = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    flatness = np.mean(librosa.feature.spectral_flatness(y=y))
    rms = np.mean(librosa.feature.rms(y=y))
    try:
        f0 = librosa.pyin(y, fmin=50, fmax=300, sr=sr)[0]
        f0_clean = f0[~np.isnan(f0)]
        if len(f0_clean) > 0:
            f0_mean, f0_std, f0_min, f0_max = (np.mean(f0_clean), np.std(f0_clean),
                                               np.min(f0_clean), np.max(f0_clean))
        else:
            f0_mean = f0_std = f0_min = f0_max = 0.0
    except Exception:
        f0_mean = f0_std = f0_min = f0_max = 0.0
    mfcc_mean = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=5), axis=1)
    return np.array([spec_cent, spec_bw, spec_roll, zcr, flatness, rms,
                     f0_mean, f0_std, f0_min, f0_max, *mfcc_mean])

# smoke test on ONE file before the long loop
_test_vid = next(iter(val_eval if val_eval else train_eval))
_f = extract_features(audio_files[_test_vid])
assert _f is not None and _f.shape == (15,), f"smoke test failed: {_f}"
print(f"Smoke test OK: {_test_vid} -> shape {_f.shape}")

In [ ]:
# Cell 4: Load model + checkpoint-safe evaluation loop
# Checkpoint after every N videos so a Colab disconnect never loses progress
import torch, torch.nn as nn
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm import tqdm

CKPT = f"{OUT_DIR}/eval_checkpoint.json"
CHECKPOINT_EVERY = 25

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(15, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 1), nn.Sigmoid())
    def forward(self, x):
        return self.net(x)

MODEL_PATH = f"{BASE}/models/top200_prosody_model.pt"
if not Path(MODEL_PATH).exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH} — check Drive/models/")
model = Net()
sd = torch.load(MODEL_PATH, map_location='cpu')
missing, unexpected = model.load_state_dict(sd, strict=False)
if missing:
    print(f"[warn] missing keys: {missing}")
model.eval()
print(f"Model loaded: {MODEL_PATH}")

def video_label(vid):
    """1 if the video has any 'risa' (laugh) segment, else 0."""
    try:
        df = pd.read_csv(label_files[vid])
    except Exception:
        return None
    lbl_col = [c for c in df.columns if 'label' in c.lower() or 'lab' in c.lower()]
    if not lbl_col:
        return None
    return int((df[lbl_col[0]].astype(str) == 'risa').any())

# Resume from checkpoint if present
records = []
done = set()
if Path(CKPT).exists():
    records = json.loads(Path(CKPT).read_text())
    done = {r['vid'] for r in records}
    print(f"Resuming: {len(done)} videos already done")

eval_videos = sorted(list(val_eval | train_eval))
todo = [v for v in eval_videos if v not in done]
print(f"Evaluating {len(todo)} of {len(eval_videos)} videos...")

for i, vid in enumerate(tqdm(todo)):
    lab = video_label(vid)
    if lab is None:
        continue
    feats = extract_features(audio_files[vid])
    if feats is None:
        continue
    with torch.no_grad():
        prob = model(torch.tensor(feats, dtype=torch.float32).unsqueeze(0)).item()
    records.append({'vid': vid, 'prob': prob, 'pred': int(prob > 0.5), 'label': lab,
                    'split': 'val' if vid in val_ids else 'train'})
    if (i + 1) % CHECKPOINT_EVERY == 0:
        Path(CKPT).write_text(json.dumps(records))

Path(CKPT).write_text(json.dumps(records))
all_true = [r['label'] for r in records]
all_pred = [r['pred'] for r in records]

print(f"\n=== FINAL RESULTS ({len(records)} videos) ===")
print(f"F1:         {f1_score(all_true, all_pred):.4f}")
print(f"Precision:  {precision_score(all_true, all_pred):.4f}")
print(f"Recall:     {recall_score(all_true, all_pred):.4f}")

# per-split metrics
for split in ('val', 'train'):
    t = [r['label'] for r in records if r['split'] == split]
    p = [r['pred'] for r in records if r['split'] == split]
    if t:
        print(f"{split}: F1={f1_score(t, p):.4f} (n={len(t)})")

In [ ]:
# Cell 5: Save final results + per-video predictions to Drive
results = {
    'f1': f1_score(all_true, all_pred),
    'precision': precision_score(all_true, all_pred),
    'recall': recall_score(all_true, all_pred),
    'n_videos': len(records),
    'n_val': sum(r['split'] == 'val' for r in records),
    'n_train': sum(r['split'] == 'train' for r in records),
    'pos_rate': float(np.mean(all_true)),
    'pred_pos_rate': float(np.mean(all_pred)),
}
with open(f"{OUT_DIR}/eval_results.json", 'w') as f:
    json.dump(results, f, indent=2)
pd.DataFrame(records).to_csv(f"{OUT_DIR}/per_video_predictions.csv", index=False)
print(f"Saved: {OUT_DIR}/eval_results.json")
print(f"Saved: {OUT_DIR}/per_video_predictions.csv")
print(json.dumps(results, indent=2))